# ML-Based Linux Vulnerability Detection: Preprocessing & EDA

**Goal of this notebook:** prepare the labelled CVE dataset that trains the *risk-prioritisation model* of the project (the model ranks vulnerabilities found on a Linux host by how likely they are to be exploited).

**Pipeline stage covered:** *External threat data -> CVE knowledge base -> Preprocessing & features* (boxes 3 and 5 of the architecture diagram).

| Source | What it gives us | Role |
|---|---|---|
| NVD API 2.0 | CVE description, CVSS v3 vector, CWE, affected CPEs, references | Features |
| FIRST EPSS | Probability of exploitation in the next 30 days | Baseline to compare against (not a model input) |
| CISA KEV | CVEs confirmed as exploited in the wild | Target label `in_kev` |

**How to run:** `Run All`. The first run downloads the data into a local `data/` folder (about 5 minutes for 2 years of CVEs, faster with a free NVD API key in the `NVD_API_KEY` environment variable). Later runs reuse the cache.

In [ ]:
import json, os, time, math
from collections import Counter
from pathlib import Path
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

# ---------------- configuration ----------------
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

DAYS_BACK = 730            # how many days of recently published CVEs to pull (use 120 for a quick test)
FORCE_REFRESH = False      # True = ignore the cached files and download again
NVD_API_KEY = os.getenv("NVD_API_KEY")   # optional, makes the download ~8x faster

NVD_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"
EPSS_URLS = ["https://epss.empiricalsecurity.com/epss_scores-current.csv.gz",
             "https://epss.cyentia.com/epss_scores-current.csv.gz"]
KEV_URL = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.csv"

NVD_CACHE = DATA_DIR / "nvd_raw.json"
EPSS_CACHE = DATA_DIR / "epss_scores.csv.gz"
KEV_CACHE = DATA_DIR / "kev.csv"

## 1. Data collection

NVD limits one query to a 120-day publication window and 2,000 records per page, so the function below walks backwards in 119-day windows and pages through each one. Everything is cached to disk so the (slow) download happens only once.

In [ ]:
def fetch_nvd_window(start, end, session):
    headers = {"apiKey": NVD_API_KEY} if NVD_API_KEY else {}
    pause = 0.8 if NVD_API_KEY else 6.5          # NVD rate limit: 5 req / 30 s without a key
    records, idx = [], 0
    while True:
        params = {
            "pubStartDate": start.strftime("%Y-%m-%dT%H:%M:%S.000"),
            "pubEndDate": end.strftime("%Y-%m-%dT%H:%M:%S.000"),
            "resultsPerPage": 2000,
            "startIndex": idx,
        }
        for attempt in range(5):
            r = session.get(NVD_URL, params=params, headers=headers, timeout=60)
            if r.status_code == 200:
                break
            time.sleep(10 * (attempt + 1))        # back off on 403/429/503
        else:
            r.raise_for_status()
        payload = r.json()
        batch = payload.get("vulnerabilities", [])
        records.extend(v["cve"] for v in batch)
        idx += len(batch)
        if not batch or idx >= payload.get("totalResults", 0):
            break
        time.sleep(pause)
    return records


def fetch_nvd(days_back):
    end = datetime.now(timezone.utc).replace(tzinfo=None)
    n_windows = math.ceil(days_back / 119)
    all_records = []
    with requests.Session() as s:
        for i in range(n_windows):
            w_end = end - timedelta(days=119 * i)
            w_start = w_end - timedelta(days=119)
            got = fetch_nvd_window(w_start, w_end, s)
            print(f"window {i + 1}/{n_windows}: {w_start:%Y-%m-%d} -> {w_end:%Y-%m-%d}  {len(got)} CVEs")
            all_records.extend(got)
    return all_records


def download_first_working(urls, dest):
    last_err = None
    for u in urls if isinstance(urls, list) else [urls]:
        try:
            r = requests.get(u, timeout=180)
            r.raise_for_status()
            dest.write_bytes(r.content)
            return u
        except Exception as e:                    # try the next mirror
            last_err = e
    raise RuntimeError(f"Could not download {dest.name}: {last_err}. "
                       f"Download it manually and save it as {dest}.")

In [ ]:
# ---- NVD ----
if NVD_CACHE.exists() and not FORCE_REFRESH:
    nvd_records = json.loads(NVD_CACHE.read_text())
    print(f"Loaded {len(nvd_records):,} CVEs from cache")
else:
    nvd_records = fetch_nvd(DAYS_BACK)
    NVD_CACHE.write_text(json.dumps(nvd_records))
    print(f"Downloaded {len(nvd_records):,} CVEs")

# ---- EPSS ----
if FORCE_REFRESH or not EPSS_CACHE.exists():
    print("EPSS source:", download_first_working(EPSS_URLS, EPSS_CACHE))
epss = pd.read_csv(EPSS_CACHE, comment="#")     # first line is a '#model_version...' comment
print("EPSS rows:", len(epss))

# ---- CISA KEV ----
if FORCE_REFRESH or not KEV_CACHE.exists():
    print("KEV source:", download_first_working(KEV_URL, KEV_CACHE))
kev = pd.read_csv(KEV_CACHE)
print("KEV rows:", len(kev))

## 2. Flatten the raw NVD JSON into a table

The NVD record is deeply nested. We keep only fields that are available *at scan time* for any CVE (CVSS vector components, weakness type, affected-product info, reference tags, description text) so the trained model can later score a CVE matched on a user's machine.

In [ ]:
def pick_cvss(metrics):
    # prefer CVSS v3.1, then v3.0; the 'Primary' (NVD) entry first
    for key in ("cvssMetricV31", "cvssMetricV30"):
        entries = metrics.get(key) or []
        if entries:
            m = sorted(entries, key=lambda x: x.get("type") != "Primary")[0]
            d = m["cvssData"]
            return {
                "cvss_version": d.get("version"),
                "cvss_base": d.get("baseScore"),
                "attack_vector": d.get("attackVector"),
                "attack_complexity": d.get("attackComplexity"),
                "privileges_required": d.get("privilegesRequired"),
                "user_interaction": d.get("userInteraction"),
                "scope": d.get("scope"),
                "conf_impact": d.get("confidentialityImpact"),
                "integ_impact": d.get("integrityImpact"),
                "avail_impact": d.get("availabilityImpact"),
                "exploitability_score": m.get("exploitabilityScore"),
                "impact_score": m.get("impactScore"),
            }
    return {}


def cpe_stats(configurations):
    parts, vendors = [], []
    for cfg in configurations or []:
        for node in cfg.get("nodes", []):
            for m in node.get("cpeMatch", []):
                if not m.get("vulnerable", False):
                    continue
                f = m.get("criteria", "").split(":")      # cpe:2.3:<part>:<vendor>:<product>:...
                if len(f) > 4:
                    parts.append(f[2])
                    vendors.append(f[3])
    return {
        "n_cpe": len(parts),
        "n_vendors": len(set(vendors)),
        "affects_os": int("o" in parts),
        "affects_app": int("a" in parts),
        "affects_hw": int("h" in parts),
        "top_vendor": Counter(vendors).most_common(1)[0][0] if vendors else None,
    }


def flatten(cve):
    desc = next((d["value"] for d in cve.get("descriptions", []) if d.get("lang") == "en"), "")
    cwes = [w["value"] for ws in cve.get("weaknesses", []) for w in ws.get("description", [])]
    refs = cve.get("references", [])
    row = {
        "cve_id": cve["id"],
        "published": cve.get("published"),
        "vuln_status": cve.get("vulnStatus"),
        "description": desc,
        "primary_cwe": cwes[0] if cwes else None,
        "n_refs": len(refs),
        "has_exploit_ref": int(any("Exploit" in r.get("tags", []) for r in refs)),
        "has_vendor_advisory": int(any("Vendor Advisory" in r.get("tags", []) for r in refs)),
    }
    row.update(pick_cvss(cve.get("metrics", {})))
    row.update(cpe_stats(cve.get("configurations")))
    return row


raw = pd.DataFrame([flatten(c) for c in nvd_records])
print(raw.shape)
raw.head()

## 3. Cleaning, label creation and merging

Steps: drop duplicates (adjacent API windows share a boundary) and *Rejected* CVEs, parse dates, merge EPSS scores, and create the label **`in_kev`** (1 = CISA confirmed exploitation in the wild).

> **Known caveat (to discuss in the report):** very recent CVEs may be exploited later but are not in KEV *yet*, so some negatives are really "not yet known". This label lag is why we later use a time-based split and report precision/recall for the positive class instead of accuracy.

In [ ]:
df = raw.drop_duplicates("cve_id").copy()
print("after de-duplication:", len(df))

df = df[df["vuln_status"] != "Rejected"].copy()
print("after dropping Rejected:", len(df))

df["published"] = pd.to_datetime(df["published"], utc=True)

# ---- EPSS ----
epss = epss.rename(columns={"cve": "cve_id", "percentile": "epss_percentile"})
df = df.merge(epss[["cve_id", "epss", "epss_percentile"]], on="cve_id", how="left")

# ---- KEV label ----
kev_ids = set(kev["cveID"].str.strip())
df["in_kev"] = df["cve_id"].isin(kev_ids).astype(int)

print(f"CVEs: {len(df):,} | KEV positives: {df['in_kev'].sum():,} ({df['in_kev'].mean():.2%})")

## 4. Missing values

Understanding *why* values are missing matters more than filling them. Newly published CVEs are often not yet analysed by NVD (no CVSS vector) or not yet scored by EPSS.

In [ ]:
missing = (df.isna().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0]
display(missing.round(2).to_frame("% missing"))

fig, ax = plt.subplots(figsize=(8, 4))
missing.head(15).sort_values().plot.barh(ax=ax, color="#2A7F8E")
ax.set_xlabel("% missing")
ax.set_title("Columns with missing values")
plt.tight_layout()
plt.show()

## 5. Feature engineering and imputation

| Feature group | Engineering |
|---|---|
| CVSS numeric | keep `cvss_base`, `exploitability_score`, `impact_score`; add `has_cvss` flag; median-impute |
| CVSS vector | categorical (attack vector, complexity, privileges, interaction, scope, C/I/A) -> one-hot, missing -> `UNKNOWN` |
| Weakness | `primary_cwe` -> keep top 15 CWEs, rest `OTHER` |
| Vendor | `top_vendor` -> keep top 15 vendors, rest `OTHER` |
| Time | `age_days` = days between publication and the newest CVE in the dataset |
| Text | description length + keyword flags (RCE, privilege escalation, auth bypass, injection, overflow); TF-IDF can be added later |
| Product exposure | `n_cpe`, `n_vendors`, `affects_os/app/hw`, reference counts, exploit / advisory reference tags |

**Leakage rule:** `epss`, `epss_percentile` and `in_kev` are *not* model inputs. EPSS is kept only as an external baseline to compare our model against.

In [ ]:
def top_k_or_other(series, k=15):
    s = series.fillna("UNKNOWN")
    keep = s.value_counts().nlargest(k).index
    return s.where(s.isin(keep) | (s == "UNKNOWN"), "OTHER")


df["has_cvss"] = df["cvss_base"].notna().astype(int)

num_cols = ["cvss_base", "exploitability_score", "impact_score",
            "n_cpe", "n_vendors", "n_refs"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df[c] = df[c].fillna(df[c].median())

for c in ["affects_os", "affects_app", "affects_hw"]:
    df[c] = df[c].fillna(0).astype(int)

cat_cols = ["attack_vector", "attack_complexity", "privileges_required", "user_interaction",
            "scope", "conf_impact", "integ_impact", "avail_impact"]
for c in cat_cols:
    df[c] = df[c].fillna("UNKNOWN").str.upper()

df["cwe_group"] = top_k_or_other(df["primary_cwe"], 15)
df["vendor_group"] = top_k_or_other(df["top_vendor"], 15)

snapshot = df["published"].max()
df["age_days"] = (snapshot - df["published"]).dt.days
df["pub_month"] = df["published"].dt.tz_localize(None).dt.to_period("M")

df["desc_words"] = df["description"].str.split().str.len()
keywords = {
    "kw_rce": r"remote code execution|arbitrary code|execute arbitrary",
    "kw_privesc": r"privilege escalation|escalate privileges|elevat\w+ privilege",
    "kw_auth_bypass": r"authentication bypass|bypass authentication|without authentication|unauthenticated",
    "kw_injection": r"sql injection|command injection|code injection",
    "kw_overflow": r"buffer overflow|out-of-bounds|use[- ]after[- ]free|memory corruption",
}
for name, pattern in keywords.items():
    df[name] = df["description"].str.contains(pattern, case=False, regex=True, na=False).astype(int)

print("Missing values left in feature columns:",
      int(df[num_cols + cat_cols + ["age_days", "desc_words"]].isna().sum().sum()))
df[["cve_id", "cvss_base", "attack_vector", "cwe_group", "vendor_group", "age_days", "kw_rce", "in_kev"]].head()

## 6. Exploratory data analysis

### 6.1 Target balance

In [ ]:
counts = df["in_kev"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Not in KEV (0)", "In KEV (1)"], counts.values, color=["#7FA7B5", "#B5483A"])
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}\n({v / counts.sum():.2%})", ha="center", va="bottom")
ax.set_ylabel("CVE count")
ax.set_title("Class balance of the target")
ax.set_ylim(0, counts.max() * 1.18)
plt.tight_layout()
plt.show()
print("Imbalance ratio (negatives per positive):", round(counts[0] / max(counts[1], 1), 1))

**Why this matters:** with a heavy imbalance, accuracy is meaningless (predicting "never exploited" is already highly accurate). The modelling stage will use class weights or resampling and be judged on PR-AUC, recall@top-k and precision for the positive class.

### 6.2 Severity (CVSS) versus exploitation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(data=df, x="cvss_base", hue="in_kev", bins=20, stat="density",
             common_norm=False, element="step", ax=axes[0], palette=["#7FA7B5", "#B5483A"])
axes[0].set_title("CVSS base score distribution by class")

sns.boxplot(data=df, x="in_kev", y="cvss_base", ax=axes[1], palette=["#7FA7B5", "#B5483A"], hue="in_kev", legend=False)
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(["Not in KEV", "In KEV"])
axes[1].set_title("CVSS base score by class")
plt.tight_layout()
plt.show()

print(df.groupby("in_kev")[["cvss_base", "exploitability_score", "impact_score"]].describe().T.round(2))

### 6.3 Which attack characteristics are over-represented among exploited CVEs?

In [ ]:
def kev_rate(col, min_count=1, top=None):
    g = df.groupby(col)["in_kev"].agg(rate="mean", n="size", positives="sum")
    g = g[g["n"] >= min_count].sort_values("rate", ascending=False)
    return g.head(top) if top else g


fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, col in zip(axes, ["attack_vector", "privileges_required", "user_interaction"]):
    g = kev_rate(col).reset_index()
    sns.barplot(data=g, x=col, y="rate", ax=ax, color="#2A7F8E")
    for i, r in g.iterrows():
        ax.text(i, r["rate"], f"n={int(r['n']):,}", ha="center", va="bottom", fontsize=8)
    ax.set_ylabel("share of CVEs in KEV")
    ax.set_title(f"KEV rate by {col}")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
g = kev_rate("cwe_group", min_count=20).reset_index()
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=g, y="cwe_group", x="rate", ax=ax, color="#C98A1B")
ax.set_xlabel("share of CVEs in KEV")
ax.set_title("KEV rate by weakness type (CWE), groups with >= 20 CVEs")
plt.tight_layout()
plt.show()

### 6.4 EPSS as an external baseline

In [ ]:
scored = df[df["epss"].notna()]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.boxplot(data=scored, x="in_kev", y="epss", ax=axes[0], hue="in_kev", legend=False, palette=["#7FA7B5", "#B5483A"])
axes[0].set_yscale("log")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["Not in KEV", "In KEV"])
axes[0].set_title("EPSS by class (log scale)")

neg = scored[scored["in_kev"] == 0].sample(min(5000, (scored["in_kev"] == 0).sum()), random_state=42)
pos = scored[scored["in_kev"] == 1]
axes[1].scatter(neg["cvss_base"], neg["epss"], s=6, alpha=0.25, color="#7FA7B5", label="Not in KEV")
axes[1].scatter(pos["cvss_base"], pos["epss"], s=22, alpha=0.8, color="#B5483A", label="In KEV")
axes[1].set_yscale("log")
axes[1].set_xlabel("CVSS base score")
axes[1].set_ylabel("EPSS")
axes[1].set_title("CVSS vs EPSS")
axes[1].legend()
plt.tight_layout()
plt.show()

print("CVEs without an EPSS score:", int(df["epss"].isna().sum()))

### 6.5 Time trend and correlations

In [ ]:
monthly = df.groupby("pub_month").agg(cves=("cve_id", "size"), kev=("in_kev", "sum"))
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(monthly.index.astype(str), monthly["cves"], color="#7FA7B5", label="All CVEs")
ax.set_ylabel("CVEs published")
ax2 = ax.twinx()
ax2.plot(monthly.index.astype(str), monthly["kev"], color="#B5483A", marker="o", label="In KEV")
ax2.set_ylabel("KEV-listed")
ax.tick_params(axis="x", rotation=60)
ax.set_title("CVEs published per month and how many are in KEV")
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = ["cvss_base", "exploitability_score", "impact_score", "n_cpe", "n_vendors", "n_refs",
             "has_exploit_ref", "has_vendor_advisory", "affects_os", "affects_app", "age_days",
             "desc_words", "kw_rce", "kw_privesc", "kw_auth_bypass", "kw_injection", "kw_overflow", "in_kev"]
corr = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(11, 8.5))
sns.heatmap(corr, cmap="vlag", center=0, annot=True, fmt=".2f", annot_kws={"size": 7}, ax=ax)
ax.set_title("Correlation of engineered features with the target")
plt.tight_layout()
plt.show()

display(corr["in_kev"].drop("in_kev").sort_values(key=abs, ascending=False).round(3).to_frame("corr with in_kev").head(10))

**Observations to write in the report after running** (fill from your own output; do not copy numbers from anywhere else):
1. How imbalanced is the target, and what does that mean for the choice of metric?
2. Does higher CVSS severity actually separate exploited from non-exploited CVEs, or is the overlap large? (This is the core argument for adding ML on top of plain CVSS.)
3. Which attack vector / privilege / CWE groups have the highest KEV rate?
4. How closely does EPSS agree with KEV, and what is the gap our model can try to close for Linux-relevant CVEs?
5. Which features are strongly correlated with each other (redundancy) or with the label?

## 7. Build the model-ready dataset and a leak-free split

We one-hot encode the categorical features, keep `epss` out of the inputs, and split **by publication date** (older CVEs train, newer CVEs test). A random split would let the model see future CVEs and inflate results.

In [ ]:
numeric_features = ["cvss_base", "exploitability_score", "impact_score", "has_cvss",
                    "n_cpe", "n_vendors", "n_refs", "has_exploit_ref", "has_vendor_advisory",
                    "affects_os", "affects_app", "affects_hw", "age_days", "desc_words"] + list(keywords)
categorical_features = cat_cols + ["cwe_group", "vendor_group"]

X = pd.concat(
    [df[numeric_features].astype(float),
     pd.get_dummies(df[categorical_features], prefix=categorical_features, dtype=int)],
    axis=1,
)
y = df["in_kev"]
assert not {"epss", "epss_percentile", "in_kev"} & set(X.columns), "leakage columns in X"

cutoff = df["published"].quantile(0.8)
train_mask = df["published"] <= cutoff
X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

print(f"Feature matrix: {X.shape[1]} features")
print(f"Train: {len(X_train):,} CVEs ({y_train.sum()} positives) up to {cutoff:%Y-%m-%d}")
print(f"Test : {len(X_test):,} CVEs ({y_test.sum()} positives) after that date")

out = pd.concat([df[["cve_id", "published", "epss", "epss_percentile"]], X, y], axis=1)
out.to_csv(DATA_DIR / "processed_cve_features.csv", index=False)
print("Saved", DATA_DIR / "processed_cve_features.csv")

## 8. Next steps (not part of this partial implementation)

1. Train a Random Forest / Gradient Boosting classifier with class weights; evaluate PR-AUC and recall@top-k on the time-based test set; compare against CVSS-only and EPSS-only rankings.
2. Train an Isolation Forest on host configuration / open-port features collected by the scanner (config-drift anomaly detection).
3. Connect the trained risk model to the scanner: CVEs matched on a host are scored, combined with exposure (listening port, privilege of the service) and ranked in the final report.